In [1]:
import os
import csv
import re
import pdfplumber
import argparse
import io
import time
import pymupdf
import fitz
import pandas as pd
import json
from datetime import datetime
from openai import AzureOpenAI
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from typing import List
from io import StringIO

endpoint = "https://test-1-redcap.openai.azure.com/"
model_name = "gpt-4o"
deployment = "test-1-redcap"

api_version = "2024-12-01-preview"
variable_entorno = os.environ.get("AZURE_OPENAI_API_KEY")
# print(variable_entorno)
subscription_key = "<AZURE_OPENAI_API_KEY>"

#if not os.environ.get("AZURE_OPENAI_API_KEY"):
  #os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter API key for Azure: ")

In [2]:
#definicion de las funciones
def extract_text_with_pdfplumber(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text(x_tolerance=1, y_tolerance=1)# Extrae texto con preservación de layout básico
                if page_text:
                    text += page_text + "\n\n"
    except Exception as e:
        print(f"Error al procesar PDF: {str(e)}")
    return text.strip()

def clean_text(text):
    text = text.replace("\xa0", " ")
    
    text = re.sub(r"[ \t]+", " ", text) # colapsa espacios y tabs repetidos
    
    text = re.sub(r"\n\s*\n+", "\n", text) # normaliza saltos de línea múltiples a un solo \n
    return text.strip()
def extract_text_with_pymupdf(pdf_path):
    text = ""
    try:
        #abre el documento PDF
        doc = fitz.open(pdf_path)
        
        for page in doc:
            #extrae texto con opciones básicas de formato
            page_text = page.get_text("text")
            if page_text:
                page_clean = clean_text(page_text)
                text += page_clean + "\n\n"
    except Exception as e:
        print(f"Error al procesar PDF: {str(e)}")
    return text.strip()

#definicion de la funcion con la que se crea una archivo csv para ir guardando la info en un mismo archivo 
def write_to_csv(data, output_file):
    data_clean = re.sub(r'^```(?:csv)?|\s*```$', '', data, flags=re.IGNORECASE).strip() #limpia la data cruda que se genera
    lines = data_clean.split('\n')
    if len(lines) < 2:
        raise ValueError("Los datos no contienen suficientes líneas (se espera al menos encabezado y una fila de datos).")
    header = lines[0].split(',')
    data_row = lines[1].split(',')
     # Verifica si el archivo existe para determinar si hay que escribir los encabezados
    write_header = not os.path.exists(output_file)
    with open(output_file, 'a', newline='', encoding='utf-8-sig') as csvfile:
        writer = csv.writer(csvfile)
        if write_header:
            writer.writerow(header)
            
        writer.writerow(data_row)


#definicion de la funcion para crear una lista con los path de los archivos contenidos en la carpeta de interes               
def get_pdf_path(folder):
    pdf_paths = []
    for archivo in os.listdir(folder):
        if archivo.lower().endswith('.pdf'):
            path_completo = os.path.join(folder, archivo)
            pdf_paths.append(path_completo)
    return pdf_paths

def write_json_to_csv(json_data, output_file):
    encabezados = [
        "record_id", "provincia_rx01", "fecha_nacimiento_rx01", "nombre_paciente_rx00_1", 
        "apellido_paciente_rx00_1", "hospital_dx01", "lateralidad_lesion_dx01", 
        "clin_glinf_axilar_dx01", "metastasis_lesion_dx01", "ubicacion_lesion_dx01", 
        "metodo_eval_dx02", "lateralidad_dx02", "ubicacion_cuadrante_dx02", 
        "fecha_cirugia_cx01", "tipo_operacion_cx01", "tipo_histologia_dx02___1", 
        "tipo_histologia_dx02___2", "tipo_histologia_dx02___3", "tipo_histologia_dx02___4", 
        "tipo_histologia_dx02___5", "tipo_histologia_dx02___6", "tipo_histologia_dx02___7", 
        "tipo_histologia_dx02___8", "tipo_histologia_dx02___9", "tipo_histologia_dx02___10", 
        "tipo_histologia_dx02___11", "tipo_histologia_dx02___12", "tipo_histologia_dx02___13", 
        "tipo_histologia_dx02___14", "tipo_histologia_dx02___15", "tipo_histologia_dx02___16", 
        "tipo_histologia_dx02___17", "tipo_histologia_dx02___18"
    ]
    write_header = not os.path.exists(output_file)
        
    with open(output_file, 'a', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        if write_header:
            writer.writerow(encabezados)
        row = [json_data.get(campo, 'uncertain') for campo in encabezados]
        writer.writerow(row)

In [3]:
#se busca analizar los pdfs todos al mismo tiempo de manera que luego se pueda generar un solo archivo csv con la info de los que se tiene hasta el momento
pdf_folder = "C:/Users/Santi/Downloads/folder-test"
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
output_file = f"tabla1_temp1.0__{timestamp}.csv"
path_csv = f"tabla1_temp1.0__{timestamp}.csv"
#se obtiene una lista de los path de los archivos contenidos en la carpeta que se le paso antes
pdf_files_path = get_pdf_path(pdf_folder)
print(pdf_files_path)

['C:/Users/Santi/Downloads/folder-test\\historiaclinica1.pdf', 'C:/Users/Santi/Downloads/folder-test\\historiaclinica2.pdf', 'C:/Users/Santi/Downloads/folder-test\\historiaclinica3.pdf', 'C:/Users/Santi/Downloads/folder-test\\historiaclinica4.pdf', 'C:/Users/Santi/Downloads/folder-test\\historiaclinica5.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica2.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica3.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica4.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica5.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica6.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica7.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica8.pdf', 'C:/Users/Santi/Downloads/folder-test\\Impresion Historia Clinica9.pdf', 'C:/Users/Santi/Downloads/folder-

In [4]:
#se crea una lista en donde se va a almacenar la informacion que se extraiga de los archivos
#se genera un ciclo for en donde se recorre la lista con los path de las HC y se llama a la funcion definida para extraer la info
#por ultimo con el comando .append se van 'pegando' los textos que se extraen en la misma lista
extracted_text_list = []
for file in pdf_files_path:
    extracted_text = extract_text_with_pymupdf(file)
    if extracted_text:  # Check if extraction was successful
        #print(extracted_text)
        extracted_text_list.append(extracted_text)
        

In [ ]:
# Configuración del sistema para procesamiento estructurado
system_prompt = """Eres un especialista en medicina y en extracción de datos de historias clínicas. Vas a extraer información en
formato JSON a partir de los archivos que se carguen como entrada con exactamente 33 columnas y exactamente
los mismos encabezados que se detallan a continuacion:
[record_id,provincia_rx01,fecha_nacimiento_rx01,nombre_paciente_rx00_1,apellido_paciente_rx00_1,hospital_dx01,
lateralidad_lesion_dx01,clin_glinf_axilar_dx01,metastasis_lesion_dx01,ubicacion_lesion_dx01,metodo_eval_dx02,lateralidad_dx02,
ubicacion_cuadrante_dx02,fecha_cirugia_cx01,tipo_operacion_cx01,tipo_histologia_dx02___1,tipo_histologia_dx02___2,
tipo_histologia_dx02___3,tipo_histologia_dx02___4,tipo_histologia_dx02___5,tipo_histologia_dx02___6,tipo_histologia_dx02___7,
tipo_histologia_dx02___8,tipo_histologia_dx02___9,tipo_histologia_dx02___10,tipo_histologia_dx02___11,tipo_histologia_dx02___12,
tipo_histologia_dx02___13,tipo_histologia_dx02___14,tipo_histologia_dx02___15,tipo_histologia_dx02___16,tipo_histologia_dx02___17,
tipo_histologia_dx02___18]

Tu tarea es:
1. Extraer la información relevante según las preguntas que el usuario formuló en user_questions.
2. DEBES generar un objeto JSON valido con las 33 columnas requeridas, con un objeto por cada 
archivo analizado.
3. Validar que las fechas tengan el formato DD/MM/YYYY.
4. Buscar responder todos los campos.
5. NO agregues variables extras ni tampoco elimines variables. NO inventes datos. Responnde una y solo una de las columnas**
6. Devuelve solamente los encabezados junto con los datos extraidos. En caso de tener mas de un diagnostico en la historia clinica
hacer foco en el diagnostico de cancer de mama. 
7. NO agregues texto extra, unicamente las variables con sus encabezado. Sino encuentras la informacion requerida colocar 'uncertain'

METODOLOGÍA:
1. Análisis exhaustivo: examinar TODO el contenido de cada archivo para localizar la información requerida, 
especificamente lo referido a los diagnosticos de cáncer de mama
2. Validación cruzada: verificar cada dato en múltiples secciones del documento siempre que sea posible
3. Control de calidad:
   - Verificar que cada campo corresponda al tipo de dato esperado
   - Confirmar coherencia entre campos relacionados (ej: lateralidad en dx01 y dx02)

ESTRUCTURA JSON REQUERIDA:
{
  "record_id": "valor",
  "provincia_rx01": "valor",
  "fecha_nacimiento_rx01": "valor",
  "nombre_paciente_rx00_1": "valor",
  "apellido_paciente_rx00_1": "valor",
  "hospital_dx01": "valor",
  "lateralidad_lesion_dx01": "valor",
  "clin_glinf_axilar_dx01": "valor",
  "metastasis_lesion_dx01": "valor",
  "ubicacion_lesion_dx01": "valor",
  "metodo_eval_dx02": "valor",
  "lateralidad_dx02": "valor",
  "ubicacion_cuadrante_dx02": "valor",
  "fecha_cirugia_cx01": "valor",
  "tipo_operacion_cx01": "valor",
  "tipo_histologia_dx02___1": "valor",
  "tipo_histologia_dx02___2": "valor",
  "tipo_histologia_dx02___3": "valor",
  "tipo_histologia_dx02___4": "valor",
  "tipo_histologia_dx02___5": "valor",
  "tipo_histologia_dx02___6": "valor",
  "tipo_histologia_dx02___7": "valor",
  "tipo_histologia_dx02___8": "valor",
  "tipo_histologia_dx02___9": "valor",
  "tipo_histologia_dx02___10": "valor",
  "tipo_histologia_dx02___11": "valor",
  "tipo_histologia_dx02___12": "valor",
  "tipo_histologia_dx02___13": "valor",
  "tipo_histologia_dx02___14": "valor",
  "tipo_histologia_dx02___15": "valor",
  "tipo_histologia_dx02___16": "valor",
  "tipo_histologia_dx02___17": "valor",
  "tipo_histologia_dx02___18": "valor"
}

PROTOCOLOS ESPECÍFICOS:
- Para nombres y apellidos, colocar las iniciales en mayuscula y el resto en minuscula
- Para hospitales: verificar todas las menciones del centro médico en documento
- Para lateralidad: confirmar en hallazgos clínicos y reportes quirúrgicos
- Para metástasis_lesion_dx01: revisar especialmente estudios de imagen y patología
- Para clinf_glinf_axilar: verificar todas las menciones acerca de los ganglios axilares si son palpables o no, tomar
    los casos que se mencionan ganglios axilares inespecificos como no palpable y los que mencionan que no se palpan adenopatias 
    tambien como no palpable. Por otro lado, tomar que si se menciona adenomegalia o axila positiva o ganglio con
    corteza engrosada se debe tomar como ganglio palpable positivo. Debes colocar la informacion presente en el ultimo informe del examen fisico
- Para ubicacion_lesion_dx01: revisar los informes los informes de mamografia, de ecografia mamaria/ecografico, los reportes de exámenes físicos, 
    exámenes mamarios, los reportes quirúrgicos, los informes de anatomía patológica. Buscar en los informes la ubicacion por 
    las frases: hs 1 / hs 12 / hs 10 / hs 9 / hs 5 / etc., y segun esa informacion asignar el valor estipulado. 
    Considerar si se menciona multicentrico como ubicacion de lesion equivalente a "porcion central = 3"
- Para metodo_eval_dx02: colocar el tipo de biopsia o lo presente en los informes de anatomía patológica, los informes de clinica quirurgica.
- Para ubicacion_cuadrante_dx02: evaluar los informes de mamografia, los reportes de exámenes físicos, los reportes.
   quirúrgicos, los informes de anatomía patológica, los informes de clinica quirurgica, y segun esa informacion asignarle el valor 
   estipulado.
- Para tipo_operacion_cx01: evaluar el informe de clinica quirurgica, y colocar el procedimiento quirurgico programado.
    Validarlo con menciones que se hagan en el resto del documento. Tomar la frase "se opera de biopsia incisional" como OTRA 
- Para tipo_histologia_dx02: Generar EXACTAMENTE 18 columnas, ni más ni menos, asignar valores binarios (1=correcto/0=incorrecto) segun la columna.
   Revisar especialmente los informes de anatomia patologica en busca de diagnostico / tipo de histologia encontrado. 
   Considerar diagnosticos como fibroadenomas, como otros tipos histologico que corresponde a la columna 18. No agregar variables 
   extras, limitarse a responder las 18 opciones. NO crear variables adicionales fuera de las 18 especificadas en user questions

Siempre devuelve la respuesta en formato JSON, sin texto adicional, ni caracteres especiales adicionales. 
"""

user_questions = """
Extraer las siguientes 33 columnas exactamente con estos encabezados, en este orden, sin agregar variables extras en las filas 
analizadas:

[record_id,provincia_rx01,fecha_nacimiento_rx01,nombre_paciente_rx00_1,apellido_paciente_rx00_1,hospital_dx01,
lateralidad_lesion_dx01,clin_glinf_axilar_dx01,metastasis_lesion_dx01,ubicacion_lesion_dx01,metodo_eval_dx02,lateralidad_dx02,
ubicacion_cuadrante_dx02,fecha_cirugia_cx01,tipo_operacion_cx01,tipo_histologia_dx02___1,tipo_histologia_dx02___2,
tipo_histologia_dx02___3,tipo_histologia_dx02___4,tipo_histologia_dx02___5,tipo_histologia_dx02___6,tipo_histologia_dx02___7,
tipo_histologia_dx02___8,tipo_histologia_dx02___9,tipo_histologia_dx02___10,tipo_histologia_dx02___11,tipo_histologia_dx02___12,
tipo_histologia_dx02___13,tipo_histologia_dx02___14,tipo_histologia_dx02___15,tipo_histologia_dx02___16,tipo_histologia_dx02___17,
tipo_histologia_dx02___18]

VALORES A ASIGNAR:
- record_id (colocar un valor numérico igual a 1 para el primer analisis)
- provincia_rx01 (provincia donde se atiende la paciente, colocar un 1 si es Salta, o un 2 si es Misiones)
- fecha_nacimiento_rx01 (fecha de nacimiento)
- nombre_paciente_rx00_1 (Nombre de la paciente, poner primer y segundo nombre)
- apellido_paciente_rx00_1 (Apellido de la paciente)
- hospital_dx01: 
  1 = Hospital Oñativia | 2 = Hospital Materno Infantil | 3 = Hospital Madariaga
- lateralidad_lesion_dx01 : (Lateralidad: izquierda/derecha, si es izquierda colocar un 1, si es derecha, un 2.) 
  1 = Izquierda | 2 = Derecha
- clin_glinf_axilar_dx01: (¿los ganglios axilares son palpables? si es afirmativo colocar un 1, caso contrario colocar un 2)
  1 = Ganglios axilares palpables | 2 = Ganglios axilares NO palpables
- metastasis_lesion_dx01: (¿hay metastasis?) 
  1 = Si | 2 = NO
- ubicacion_lesion_dx01 (ubicación de la enfermedad: de acuerdo a la siguiente lista asignar el número que corresponda a la ubicación 
de la enfermedad: 
      1 = Cola axilar | 2 = pezón | 3 = Porción central (Subareolar) | 4 = Hs. 12:00 | 5 = Hs. 1:00 | 6 = Hs. 2:00 | 7 = Hs. 3:00 |
      8 = Hs. 4:00 | 9 = Hs. 5:00 | 10 = Hs. 6:00 | 11 = Hs. 7:00 | 12 = Hs. 8:00 | 13 = Hs. 9:00 | 14 = Hs. 10:00 | 15 = Hs. 11:00) 
- metodo_eval_dx02: Tipo de biopsia: asignar el metodo de evaluacion que fue usado de acuerdo al siguiente listado: 
  1 = Biopsia con aguja gruesa o biopsia tru-cut, 
  2 = Biopsia incisional, 
  3 = Tumorectomía, 
  4 = Mastectomía radical modificada, 
  5 = Mastectomia simple.
- lateralidad_dx02: Lateralidad: izquierda/derecha, si es izquierda colocar un 1, si es derecha, un 2.)
- ubicacion_cuadrante_dx02: Colocar la ubicación del cuadrante en donde se ubica la lesión, según la siguiente lista: 
      1 = Porción central (pezón/subareolar),
      2 = Superior interno,
      3 = Superior externo,
      4 = Inferior interno,
      5 = Inferior externo,
      6 = Mama entera,
      7 = Compromiso de más de un cuadrante
- fecha_cirugia_cx01: colocar la fecha en que se realizó la cirugia de cáncer de mama - verificar que sea la fecha que coincide con la 
  foja quirurgica
- tipo_operacion_cx01: Tipo de operación-cirugía, colocar qué tipo de cirugia se realizó según la siguiente lista: 
  1 = Mastectomía radical modificada, 2 = Mastectomía simple, 3 = Cuadrantectomía, 4 = Tumorectomía, 5 = Otra)
- tipo de histologia: asignar el tipo histológico asociado al diagnóstico de cáncer de mama, de acuerdo a la siguiente lista 
(una sola opcion debe marcarse como correcta con '1', las otras 17 variables debe colocarse un '0'. Limita las respuestas a las 
18 opciones/columnas especificadas, que no queden respuestas sin encabezado):
  tipo_histologia_dx02___1 = Sin carcinoma invasor residual,
  tipo_histologia_dx02___2 = Carcinoma invasor sin tipo especial/NST (ductal),
  tipo_histologia_dx02___3 = Carcinoma micro-invasor,
  tipo_histologia_dx02___4 = Carcinoma lobulillar invasor,
  tipo_histologia_dx02___5 = Carcinoma invasor con características mixtas ductales y lobulillares,
  tipo_histologia_dx02___6 = Carcinoma tubular,
  tipo_histologia_dx02___7 = Carcinoma cribiforme invasor,
  tipo_histologia_dx02___8 = Carcinoma mucinoso,
  tipo_histologia_dx02___9 = Carcinoma micro-papilar invasor,
  tipo_histologia_dx02___10 = Adenocarcinoma apocrino,
  tipo_histologia_dx02___11 = Carcinoma metaplásico,
  tipo_histologia_dx02___12 = Carcinoma papilar encapsulado con invasión,
  tipo_histologia_dx02___13 = Carcinoma papilar sólido con invasión,
  tipo_histologia_dx02___14 = Adenocarcinoma papilar intraductal con invasión,
  tipo_histologia_dx02___15 = Carcinoma adenoide quístico,
  tipo_histologia_dx02___16 = Tumor neuroendocrino,
  tipo_histologia_dx02___17 = Carcinoma neuroendocrino,
  tipo_histologia_dx02___18 = Otro tipo histológico no incluido.

"""


In [ ]:
for i in range(5):
#se busca analizar los pdfs todos al mismo tiempo de manera que luego se pueda generar un solo archivo csv con la info de los que se tiene hasta el momento
    pdf_folder = "C:/Users/Santi/Downloads/folder-test"
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    output_file = f"tabla{i}_temp0.8__{timestamp}.csv"
    #path_csv = f"tabla{i}_temp0.9__{timestamp}.csv"
    #se obtiene una lista de los path de los archivos contenidos en la carpeta que se le paso antes
    pdf_files_path = get_pdf_path(pdf_folder)
        #print(pdf_files_path)
    for elemento in extracted_text_list:
            #print(elemento)
                
            input_content = f"Requisitos:\n{user_questions}\n\nContenido del PDF:\n{elemento}"
            client = AzureOpenAI(
            api_version=api_version,
                azure_endpoint=endpoint,
                api_key=variable_entorno,
            )

            response = client.chat.completions.create(
                messages=[
                        {
                            "role": "system",
                            "content": system_prompt,
                        },
                        {
                            "role": "user",
                            "content": input_content,
                        }
                    ],
                    max_tokens=4096,
                    temperature=0.1,
                    top_p=1.0,
                    model=deployment,
                    response_format={"type": "json_object"}  
                )
            output_data_json = response.choices[0].message.content        
            json_data = json.loads(output_data_json)
            write_json_to_csv(json_data, output_file)
            print(output_data_json)            
    

JSONDecodeError: Expecting value: line 3 column 20 (char 41)